# NHANES Lp(a) marginal distribution

This project derives an age × sex marginal distribution of serum
lipoprotein(a) for U.S. adults from the only U.S.-nationally-
representative survey that measured it: **NHANES III Phase II
(1991–1994)**, lab data file `lab.dat`, variable `LPP` (mg/dL).

## Why only NHANES III?

Probing the cached continuous-NHANES cycles (1999–2022) and the
CDC variable lookup confirmed that **Lp(a) was not measured in any
continuous NHANES cycle**. The closest lipid files (`TRIGLY_*`,
`HDL_*`, `TCHOL_*`, `APOB_*` where present) carry only LDL / HDL /
T-Chol / triglycerides / apolipoprotein-B. NHANES III Phase II
(measured at Johns Hopkins Lipoprotein Analytical Lab, Baltimore)
is the single nationally-representative source.

An earlier comment in the project repo's `data_values.py`
claimed NHANES collected Lp(a) in 1999–2006 and 2015–2018 — that
was incorrect. Subsequent notebooks correct that.

## Notebook map

| # | Notebook | What it does |
|---|----------|--------------|
| 01 | [Download lab.dat](01_download_lab.ipynb) | Cache the NHANES III Release 1A lab data file (single fixed-width ASCII, 56 MB) and parse just the columns we need (SEQN, age, sex, phase, survey-design weights/PSU/stratum, LPP). Saves `data/derived/nhanes3_lpa.parquet`. |
| 02 | [Marginal + literature comparison](02_lpa_marginal.ipynb) | The main analysis. Survey-weighted age × sex marginal of Lp(a) with Taylor-linearized SEs; comparison against the literature point estimate that's currently used as the project stub (`mean 25 / SD 35 mg/dL`). |

## Decision the analysis informs

If age × sex variation in Lp(a) is **small** relative to the within-
stratum SD, a single scalar mean / SD (Option 3 in the project memo)
is acceptable and a future iteration can pursue more elaborate data
strategies (linkage to ARIC / MESA, cohort-specific extrapolation).
If age × sex variation is **large**, we wire NHANES III's per-
stratum mean / SD into the project loader's `load_lpa_exposure`.

Either way, the 1991–94 measurement timestamp is a recognized
limitation; Lp(a) is heavily genetic so individual values are
stable, but the U.S. *population* mix has shifted demographically
since then (notably: more racial diversity among older adults).
Lp(a) varies substantially by ancestry, so the 1991–94 marginal
may slightly under-represent today's average if Black and South
Asian shares have grown.

## Reproducing

```bash
cd ai_assisted_us_health_data_analysis/nhanes_lpa_distribution
uv venv && uv pip install -r requirements.txt
source .venv/bin/activate
for nb in 01_download_lab.ipynb 02_lpa_marginal.ipynb; do
    jupyter nbconvert --to notebook --execute --inplace "$nb"
done
```